In [0]:
import duckdb

# Path to Delta Lake table on S3
s3_delta_path = 's3://data-and-analytics-assets/__metadata/master'

# Use Databricks/Spark to read Delta Lake table
spark_df = spark.read.format('delta').load(s3_delta_path)

# Convert Spark DataFrame to Pandas
pandas_df = spark_df.limit(1000).toPandas()  # Limit for memory safety

# Use a writable path for DuckDB database file
# /tmp is always writable in Databricks
duckdb_db_path = '/tmp/duckdb_from_delta.db'

# Connect to DuckDB (persistent database file)
db = duckdb.connect(database=duckdb_db_path, read_only=False)

# Load Pandas DataFrame into DuckDB
# Create table and insert data
try:
    db.execute("DROP TABLE IF EXISTS delta_table")
    db.execute("CREATE TABLE delta_table AS SELECT * FROM pandas_df")
    print("Delta table loaded into DuckDB database.")
except Exception as e:
    print("Error loading table into DuckDB:", e)

# Query sample from DuckDB table
duckdb_sample = db.execute("SELECT * FROM delta_table LIMIT 5").fetchdf()
display(duckdb_sample)

# Close connection
db.close()